<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/49_persistent_memory_agent/persistent_memory_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
import os
import re

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
MEMORY_FILE = "feedback_memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return []

def save_memory(memory):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f)

In [ ]:
feedback_memory = load_memory()

In [ ]:
def store_feedback(query, answer):
    embedding = model.encode(query).tolist()

    feedback_memory.append({
        "query": query,
        "embedding": embedding,
        "answer": answer
    })

    save_memory(feedback_memory)

In [ ]:
def retrieve_feedback(query, threshold=0.6):
    if not feedback_memory:
        return None

    query_embedding = model.encode(query)

    similarities = []

    for item in feedback_memory:
        sim = cosine_similarity(
            [query_embedding],
            [item["embedding"]]
        )[0][0]
        similarities.append(sim)

    best_idx = np.argmax(similarities)

    if similarities[best_idx] >= threshold:
        return feedback_memory[best_idx]["answer"]

    return None

In [ ]:
def persistent_agent(query):

    # check stored memory
    feedback = retrieve_feedback(query)
    if feedback:
        return {"source": "persistent_memory", "answer": feedback}

    # fallback
    return {"source": "tool", "answer": "No knowledge available"}

In [9]:
# first run (no memory)
q1 = "Explain deep learning"
print(persistent_agent(q1))

# store feedback
store_feedback(q1, "Deep Learning is a subset of machine learning using neural networks.")

# simulate restart (reload memory)
feedback_memory = load_memory()

# second run (memory persists!)
q2 = "Explain about deep learning"
print(persistent_agent(q2))

{'source': 'tool', 'answer': 'No knowledge available'}
{'source': 'persistent_memory', 'answer': 'Deep Learning is a subset of machine learning using neural networks.'}
